In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as _sum, lag, to_date, date_format, round

spark = SparkSession.builder.appName("NetflixMoM").getOrCreate()

# Sample Data (Netflix Watch History)
data = [
    ("U1", "2024-01-01", 60),
    ("U1", "2024-01-10", 120),
    ("U1", "2024-02-05", 180),
    ("U1", "2024-03-01", 240),
    ("U2", "2024-01-01", 200),
    ("U2", "2024-03-01", 400)
]

columns = ["user_id", "watch_date", "watch_minutes"]

df = spark.createDataFrame(data, columns) \
          .withColumn("watch_date", to_date("watch_date"))

# Step 1: Extract month
df = df.withColumn("month", date_format("watch_date", "yyyy-MM"))

# Step 2: Monthly aggregation
monthly_df = df.groupBy("user_id", "month") \
               .agg(_sum("watch_minutes").alias("monthly_watch_time"))

# Step 3: Window for previous month
window_spec = Window.partitionBy("user_id").orderBy("month")

# Step 4: Calculate MoM growth

result_df = monthly_df.withColumn("prev_month_watch", lag("monthly_watch_time").over(window_spec)) \
    .withColumn(
        "mom_growth_pct",
        round(((col("monthly_watch_time") - col("prev_month_watch")) / col("prev_month_watch")) * 100, 2)
    )

result_df.show()